# EDA — Infraestructura de carga EV en Chile

Análisis exploratorio sobre estaciones de OpenChargeMap + ocupación simulada.

**Preguntas:**
1. ¿Cómo se distribuye la infraestructura por región y potencia?
2. ¿Qué operadores dominan y a qué precios?
3. ¿Cuáles son los patrones de ocupación por hora?
4. ¿Qué estaciones están saturadas en horas peak?

> ⚠️ La ocupación es sintética (no hay fuente pública en tiempo real). Ver README.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import plotly.express as px

from ev_charging import OpenChargeMapClient, ChargingAnalyzer
from ev_charging.availability import generate_occupancy

pd.set_option("display.max_columns", None)

## 1. Carga de datos

In [ ]:
client = OpenChargeMapClient()  # requiere OCM_API_KEY en ../.env
stations = client.to_dataframe(client.fetch_stations(max_results=500))
print(f"{len(stations)} estaciones descargadas")
stations.head()

In [ ]:
# Calidad de datos: nulos por columna
stations.isna().mean().sort_values(ascending=False).round(2)

## 2. Infraestructura: distribución regional y potencia

In [ ]:
analyzer = ChargingAnalyzer(stations)
analyzer.stations_by_region().head(10)

In [ ]:
analyzer.power_distribution()

In [ ]:
fig = px.scatter_map(
    stations.dropna(subset=["latitude", "longitude"]),
    lat="latitude", lon="longitude",
    color="max_power_kw", size="n_connectors",
    hover_name="name", zoom=4, map_style="open-street-map",
    title="Estaciones de carga en Chile",
)
fig.show()

## 3. Operadores y precios

In [ ]:
analyzer.operator_share().head(10)

In [ ]:
# OCM guarda precios como texto libre — parse_price_clp_kwh extrae CLP/kWh
analyzer.price_summary()

## 4. Patrones de ocupación (simulada)

In [ ]:
occupancy = generate_occupancy(stations, days=14)
analyzer = ChargingAnalyzer(stations, occupancy)

hourly = analyzer.occupancy_by_hour()
fig = px.line(hourly, title="Ocupación media por hora — semana vs fin de semana",
              labels={"value": "tasa de ocupación", "hour": "hora"})
fig.show()

In [ ]:
analyzer.peak_hours()

## 5. Insight accionable: estaciones saturadas

In [ ]:
# Ocupación media >= 80% en horas peak → candidatas a ampliación de capacidad
analyzer.saturated_stations(threshold=0.8).head(15)

## Conclusiones

- La infraestructura se concentra en la Región Metropolitana; la potencia media varía fuerte entre regiones.
- Los peaks de ocupación entre semana ocurren en horario de commute (8–10h y 18–21h); los fines de semana la curva se aplana con máximo al mediodía.
- Las estaciones con ocupación peak ≥80% son candidatas naturales a ampliación.

**Siguiente paso:** acumular snapshots reales con `save_snapshot()` vía cron y reemplazar la simulación por datos observados.